# QuantumPad · Real compute lab
Run this notebook in Colab or a Python environment with PyTorch. CPU works by default; use Runtime → Change runtime type for a GPU if your account has capacity. This notebook measures actual hardware. It does not rent resources, configure billing, or guarantee a free GPU.
Cloud runtimes are managed in the hosting provider's console. Results are workload-specific, not peak vendor performance.

In [ ]:
import torch, time, json, hashlib
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Actual execution device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('PyTorch:', torch.__version__)
torch.manual_seed(42)


In [ ]:
n = 1024
a = torch.rand(n, n, device=device)
b = torch.rand(n, n, device=device)
_ = a @ b  # Warm-up
if device.type == 'cuda': torch.cuda.synchronize()
started = time.perf_counter()
output = a @ b
if device.type == 'cuda': torch.cuda.synchronize()
elapsed = time.perf_counter() - started
reference = (a[0].double() * b[:, 0].double()).sum().item()
relative_error = abs(output[0,0].item() - reference) / max(1, abs(reference))
assert relative_error < 0.001, 'Output verification failed'
result = {'device': str(device), 'matrix_size': n, 'elapsed_ms': elapsed * 1000, 'measured_gflops': 2*n**3 / elapsed / 1e9, 'sample_relative_error': relative_error, 'output_sha256': hashlib.sha256(output.cpu().numpy().tobytes()).hexdigest()}
print(json.dumps(result, indent=2))
with open('exaflop-result.json', 'w') as file: json.dump(result, file, indent=2)


Return to [QuantumPad](https://exaflop.vercel.app/build) to compare cloud compute. Download exaflop-result.json from the notebook file browser. Stop/disconnect your runtime in the provider console when done.